# Project 3 — Phase 5 (closing): Baseline + OOD evaluation

Runs two passes in one session:
1. **Baseline** — Qwen2.5-7B-Instruct zero-shot (no adapter) on the held-out test set → `baseline.json`
2. **OOD** — LoRA v1 adapter on the 60-example synthetic clinical-note set → `lora_v1_ood.json`

Kaggle inputs required (set the paths in Cell 2):
- `BASE_MODEL`  (Qwen2.5-7B-Instruct, hub revision pinned)
- `ADAPTER_PATH`  (your existing harmony-lora-adapter dataset)
- `TEST_FILE`  (data/processed/test.jsonl)
- `OOD_FILE`  (evaluation/synthetic_ade_eval.jsonl — upload alongside test.jsonl)

Outputs to `/kaggle/working/eval_reports/`. Download both JSON files when done.


In [1]:
!pip install -q "transformers==4.46.3" "trl==0.11.4" peft bitsandbytes accelerate datasets json-repair pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 91.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 14.9 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.

In [12]:
import os, sys, json, time, datetime, torch
from pathlib import Path

BASE_MODEL          = "Qwen/Qwen2.5-7B-Instruct"
BASE_MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"

# ---- EDIT THESE PATHS to match your Kaggle dataset layout ----
ADAPTER_PATH = "/kaggle/input/datasets/keertanaks/harmony-lora-adapter"
TEST_FILE    = "/kaggle/input/datasets/keertanaks/harmony-ade-data/test.jsonl"
OOD_FILE     = "/kaggle/input/datasets/keertanaks/harmony-ade-data/synthetic_ade_eval.jsonl"

OUTPUT_DIR   = "/kaggle/working/eval_reports"
MAX_SAMPLES_BASELINE = None    # full test set (2,376). Set to e.g. 300 for a quick run.
MAX_SAMPLES_OOD      = None    # full 60 OOD examples.

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert Path(ADAPTER_PATH).exists(), f"Adapter not found: {ADAPTER_PATH}"
assert Path(TEST_FILE).exists(),    f"Test file not found: {TEST_FILE}"
assert Path(OOD_FILE).exists(),     f"OOD file not found: {OOD_FILE}"
assert torch.cuda.is_available(),   "No GPU detected"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Test:    {TEST_FILE}")
print(f"OOD:     {OOD_FILE}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"Output:  {OUTPUT_DIR}")


GPU: Tesla T4
Test:    /kaggle/input/datasets/keertanaks/harmony-ade-data/test.jsonl
OOD:     /kaggle/input/datasets/keertanaks/harmony-ade-data/synthetic_ade_eval.jsonl
Adapter: /kaggle/input/datasets/keertanaks/harmony-lora-adapter
Output:  /kaggle/working/eval_reports


In [13]:
# Inline schema + validator + metrics (no app/ imports needed on Kaggle)
from typing import Literal, Optional
from pydantic import BaseModel, Field, model_validator, ValidationError
import json_repair

class SourceSpan(BaseModel):
    start_char: int = Field(ge=0)
    end_char: int = Field(ge=0)

class Entity(BaseModel):
    entity_type: Literal["medication", "adverse_event"]
    mention: str = Field(min_length=1)
    dosage: Optional[str] = None
    linked_medication: Optional[str] = None
    evidence: str = Field(min_length=1)
    source_span: SourceSpan
    @model_validator(mode="after")
    def _span_ok(self):
        if self.source_span.end_char <= self.source_span.start_char:
            raise ValueError("end_char must be > start_char")
        return self

class ValidationFlags(BaseModel):
    json_valid: bool
    schema_valid: bool
    enum_valid: bool
    evidence_present: bool

class ExtractionResult(BaseModel):
    record_id: str
    schema_version: Literal["v1"]
    entities: list[Entity]
    relation_status: Literal["related", "not_related", "none"]
    validation: ValidationFlags
    error_reason: Optional[str] = None

def build_empty_result(record_id: str, reason: str) -> ExtractionResult:
    return ExtractionResult(
        record_id=record_id, schema_version="v1", entities=[],
        relation_status="none",
        validation=ValidationFlags(json_valid=False, schema_valid=False,
                                   enum_valid=False, evidence_present=False),
        error_reason=reason,
    )

def validate_extraction(raw_text: str, record_id: str, input_text: str) -> ExtractionResult:
    try:
        parsed = json.loads(raw_text)
        json_repaired = False
    except json.JSONDecodeError:
        try:
            parsed = json_repair.loads(raw_text)
            json_repaired = True
            if not isinstance(parsed, dict):
                return build_empty_result(record_id, reason="json_parse_failed")
        except Exception:
            return build_empty_result(record_id, reason="json_parse_failed")
    parsed["record_id"] = record_id
    parsed["validation"] = {"json_valid": True, "schema_valid": True,
                            "enum_valid": True, "evidence_present": True}
    try:
        result = ExtractionResult.model_validate(parsed)
    except ValidationError:
        return build_empty_result(record_id, reason="schema_invalid")
    if json_repaired:
        result.validation.json_valid = False  # parseable only after repair
    for e in result.entities:
        if e.evidence not in input_text:
            result.validation.evidence_present = False
            break
    return result

# ---- metrics ----
def entity_f1(pred, gold, etype):
    p = [e.mention.lower().strip() for e in pred if e.entity_type == etype]
    g = [e.mention.lower().strip() for e in gold if e.entity_type == etype]
    gr = list(g); tp = 0
    for m in p:
        if m in gr: tp += 1; gr.remove(m)
    fp = len(p) - tp; fn = len(g) - tp
    pr = tp/(tp+fp) if tp+fp else 0.0
    rc = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return {"precision": pr, "recall": rc, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

def span_iou(a, b):
    inter = max(0, min(a.end_char, b.end_char) - max(a.start_char, b.start_char))
    union = max(0, max(a.end_char, b.end_char) - min(a.start_char, b.start_char))
    return inter/union if union else 0.0

def relation_macro_f1(preds, golds):
    classes = ["related", "not_related", "none"]; per = {}
    for c in classes:
        tp = sum(1 for p,g in zip(preds,golds) if p==c and g==c)
        fp = sum(1 for p,g in zip(preds,golds) if p==c and g!=c)
        fn = sum(1 for p,g in zip(preds,golds) if p!=c and g==c)
        pr = tp/(tp+fp) if tp+fp else 0.0
        rc = tp/(tp+fn) if tp+fn else 0.0
        per[c] = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return {"macro_f1": sum(per.values())/len(classes), "per_class": per}
print("schema + validator + metrics ready")


schema + validator + metrics ready


In [14]:
print(subprocess.check_output(["ls", "/kaggle/input/datasets/keertanaks"]).decode())

harmony-ade-data
harmony-lora-adapter



In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading BASE model only (no adapter) — FP16...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    torch_dtype=torch.float16, device_map="auto",
)
model.config.use_cache = False
model.eval()
print(f"Base model loaded. Free VRAM: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB")


Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Base model loaded. Free VRAM: 8.8 GB


In [16]:
INSTRUCTION = (
    "You are a clinical information extractor. Given a clinical text, extract all\n"
    "medications and adverse events as a JSON object that follows the schema below.\n"
    "Return ONLY valid JSON. If no entity is present, return entities=[] and\n"
    "relation_status=\"none\".\n\n"
    "Return ONLY this JSON structure (no record_id, no validation block — those are added by the system):\n"
    "{\n  \"schema_version\": \"v1\",\n  \"entities\": [\n    {\n"
    "      \"entity_type\": \"medication\" | \"adverse_event\",\n"
    "      \"mention\": \"<string>\",\n      \"dosage\": \"<string>\" | null,\n"
    "      \"linked_medication\": \"<string>\" | null,\n      \"evidence\": \"<string>\",\n"
    "      \"source_span\": {\"start_char\": <int>, \"end_char\": <int>}\n    }\n  ],\n"
    "  \"relation_status\": \"related\" | \"not_related\" | \"none\"\n}"
)

def run_inference(text: str) -> str:
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\nClinical text:\n{text}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, do_sample=False, max_new_tokens=512,
                             repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def load_jsonl(path):
    return [json.loads(L) for L in open(path) if L.strip()]

def parse_row(row):
    """Return (input_text, gold_dict, record_id). Handles both chat-format and OOD format."""
    if "messages" in row:
        user = row["messages"][0]["content"]
        text = user.split("Clinical text:\n", 1)[-1].strip()
        gold = json.loads(row["messages"][1]["content"])
        rid  = row.get("id") or "row"
    else:
        text = row["text"]; gold = row["gold"]; rid = row.get("id", "ood")
    return text, gold, rid

def compute_all_metrics(results, golds_per_row, pred_rel, gold_rel, inputs, latencies, json_pre):
    n = len(results)
    pred_meds = [[e for e in r.entities if e.entity_type=="medication"] for r in results]
    pred_ades = [[e for e in r.entities if e.entity_type=="adverse_event"] for r in results]
    gold_meds = [[e for e in g if e.entity_type=="medication"] for g in golds_per_row]
    gold_ades = [[e for e in g if e.entity_type=="adverse_event"] for g in golds_per_row]
    def micro(pred_lists, gold_lists, etype):
        tp=fp=fn=0
        for p,g in zip(pred_lists, gold_lists):
            r = entity_f1(p, g, etype); tp+=r["tp"]; fp+=r["fp"]; fn+=r["fn"]
        pr = tp/(tp+fp) if tp+fp else 0.0
        rc = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
        return {"precision": pr, "recall": rc, "f1": f1, "tp": tp, "fp": fp, "fn": fn}
    drug = micro(pred_meds, gold_meds, "medication")
    ade  = micro(pred_ades, gold_ades, "adverse_event")
    rel  = relation_macro_f1(pred_rel, gold_rel)
    # span metrics — greedy alignment by (type,mention) within each row
    strict_c = lenient_c = span_total = 0
    for pr_res, g_list in zip(results, golds_per_row):
        g_remaining = list(g_list)
        for pe in pr_res.entities:
            span_total += 1
            match = None; idx = -1
            for i,ge in enumerate(g_remaining):
                if ge.entity_type == pe.entity_type and ge.mention.lower().strip() == pe.mention.lower().strip():
                    match = ge; idx = i; break
            if match is None: continue
            g_remaining.pop(idx)
            if pe.source_span.start_char == match.source_span.start_char and pe.source_span.end_char == match.source_span.end_char:
                strict_c += 1
            if span_iou(pe.source_span, match.source_span) >= 0.5:
                lenient_c += 1
    # hallucination + evidence
    total_ent = 0; hall = 0; ev_ok = 0
    for r, inp in zip(results, inputs):
        for e in r.entities:
            total_ent += 1
            if e.mention.lower() not in inp.lower(): hall += 1
            if e.evidence in inp: ev_ok += 1
    return {
        "json_valid_pre_repair":  json_pre / n if n else 0.0,
        "json_valid_post_repair": sum(1 for r in results if r.validation.json_valid or r.error_reason is None) / n if n else 0.0,
        "schema_valid": sum(1 for r in results if r.validation.schema_valid) / n if n else 0.0,
        "drug_precision": drug["precision"], "drug_recall": drug["recall"], "drug_f1": drug["f1"],
        "ade_precision": ade["precision"],   "ade_recall": ade["recall"],   "ade_f1": ade["f1"],
        "relation_f1": rel["macro_f1"], "relation_per_class": rel["per_class"],
        "hallucination_rate": hall/total_ent if total_ent else 0.0,
        "evidence_accuracy":  ev_ok/total_ent if total_ent else 1.0,
        "enum_accuracy":      sum(1 for r in results if r.validation.enum_valid) / n if n else 0.0,
        "span_f1_strict":     strict_c/span_total if span_total else 0.0,
        "span_f1_lenient":    lenient_c/span_total if span_total else 0.0,
        "latency_p50_s":      float(__import__("numpy").percentile(latencies, 50)) if latencies else 0.0,
        "latency_p95_s":      float(__import__("numpy").percentile(latencies, 95)) if latencies else 0.0,
        "n_entities_total":   total_ent,
    }

def run_eval_pass(rows, label):
    print(f"\n{'='*60}\n  EVAL PASS: {label}  (n={len(rows)})\n{'='*60}")
    results, golds_per_row, pred_rel, gold_rel, inputs, latencies = [], [], [], [], [], []
    json_pre = 0; errors = []
    t_start = time.time()
    for i, row in enumerate(rows):
        if i and i % 50 == 0:
            elapsed = time.time() - t_start
            print(f"  {i}/{len(rows)}  |  {elapsed/i:.2f}s/ex  |  ETA {(elapsed/i)*(len(rows)-i)/60:.1f} min")
        text, gold_dict, rid = parse_row(row)
        t0 = time.time(); raw = run_inference(text); latencies.append(time.time()-t0)
        try: json.loads(raw); json_pre += 1
        except: pass
        res = validate_extraction(raw, record_id=str(rid) if rid != "row" else f"{label}_{i}", input_text=text)
        gold_ents = []
        for e in gold_dict.get("entities", []):
            try:
                gold_ents.append(Entity(
                    entity_type=e["entity_type"], mention=e["mention"],
                    dosage=e.get("dosage"), linked_medication=e.get("linked_medication"),
                    evidence=e["evidence"],
                    source_span=SourceSpan(start_char=e["source_span"]["start_char"],
                                           end_char=e["source_span"]["end_char"]),
                ))
            except Exception: pass
        results.append(res); golds_per_row.append(gold_ents); inputs.append(text)
        pred_rel.append(res.relation_status); gold_rel.append(gold_dict.get("relation_status","none"))
        if res.error_reason or (res.entities and any(e.mention.lower() not in text.lower() for e in res.entities)):
            if len(errors) < 20:
                errors.append({"id": f"{label}_{i}", "input": text[:300],
                               "predicted": raw[:400], "gold": json.dumps(gold_dict)[:400],
                               "error_reason": res.error_reason})
    total_min = (time.time() - t_start)/60
    print(f"  done in {total_min:.1f} min")
    m = compute_all_metrics(results, golds_per_row, pred_rel, gold_rel, inputs, latencies, json_pre)
    return m, errors
print("inference + eval-loop ready")


inference + eval-loop ready


In [17]:
# ===== PASS 1: Baseline (no adapter) on test.jsonl =====
test_rows = load_jsonl(TEST_FILE)
if MAX_SAMPLES_BASELINE: test_rows = test_rows[:MAX_SAMPLES_BASELINE]
print(f"Baseline pass: {len(test_rows)} examples")

metrics_baseline, errors_baseline = run_eval_pass(test_rows, label="baseline")

report = {
    "model": "baseline",
    "adapter_path": None,
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "test_file": TEST_FILE,
    "n_examples": len(test_rows),
    "metrics": metrics_baseline,
    "error_analysis": errors_baseline,
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
}
out_path = f"{OUTPUT_DIR}/baseline.json"
json.dump(report, open(out_path, "w"), indent=2)
print(f"\nSaved -> {out_path}")
for k,v in metrics_baseline.items():
    if isinstance(v, float): print(f"  {k:30s} = {v:.4f}")


Baseline pass: 2376 examples

  EVAL PASS: baseline  (n=2376)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  50/2376  |  7.81s/ex  |  ETA 302.6 min
  100/2376  |  8.43s/ex  |  ETA 319.9 min
  150/2376  |  9.67s/ex  |  ETA 358.7 min
  200/2376  |  9.53s/ex  |  ETA 345.5 min
  250/2376  |  9.18s/ex  |  ETA 325.4 min
  300/2376  |  10.10s/ex  |  ETA 349.6 min
  350/2376  |  10.62s/ex  |  ETA 358.6 min
  400/2376  |  11.36s/ex  |  ETA 374.1 min
  450/2376  |  11.64s/ex  |  ETA 373.6 min
  500/2376  |  11.32s/ex  |  ETA 353.9 min
  550/2376  |  11.08s/ex  |  ETA 337.3 min
  600/2376  |  11.08s/ex  |  ETA 328.1 min
  650/2376  |  11.13s/ex  |  ETA 320.3 min
  700/2376  |  11.20s/ex  |  ETA 312.8 min
  750/2376  |  10.81s/ex  |  ETA 293.0 min
  800/2376  |  10.40s/ex  |  ETA 273.1 min
  850/2376  |  10.02s/ex  |  ETA 254.8 min
  900/2376  |  9.64s/ex  |  ETA 237.2 min
  950/2376  |  9.31s/ex  |  ETA 221.3 min
  1000/2376  |  9.01s/ex  |  ETA 206.5 min
  1050/2376  |  8.74s/ex  |  ETA 193.1 min
  1100/2376  |  8.54s/ex  |  ETA 181.7 min
  1150/2376  |  8.27s/ex  |  ETA 169.0 min
  1200/2376  |  8.0

/tmp/ipykernel_58/3725401376.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow().isoformat() + "Z",


In [18]:
# ===== Now load LoRA adapter on top of the same base model =====
from peft import PeftModel
del model
torch.cuda.empty_cache()

print("Reloading base model + LoRA adapter...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    torch_dtype=torch.float16, device_map="auto",
)
model.config.use_cache = False
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print(f"LoRA adapter loaded. Free VRAM: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB")


Reloading base model + LoRA adapter...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LoRA adapter loaded. Free VRAM: 8.5 GB


In [19]:
# ===== PASS 2: LoRA adapter on synthetic OOD set =====
ood_rows = load_jsonl(OOD_FILE)
if MAX_SAMPLES_OOD: ood_rows = ood_rows[:MAX_SAMPLES_OOD]
print(f"OOD pass: {len(ood_rows)} examples")

metrics_ood, errors_ood = run_eval_pass(ood_rows, label="ood")

report = {
    "model": "lora",
    "adapter_path": ADAPTER_PATH,
    "base_model": BASE_MODEL,
    "base_model_revision": BASE_MODEL_REVISION,
    "test_file": OOD_FILE,
    "test_set": "synthetic_ood",
    "n_examples": len(ood_rows),
    "metrics": metrics_ood,
    "error_analysis": errors_ood,
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
}
out_path = f"{OUTPUT_DIR}/lora_v1_ood.json"
json.dump(report, open(out_path, "w"), indent=2)
print(f"\nSaved -> {out_path}")
for k,v in metrics_ood.items():
    if isinstance(v, float): print(f"  {k:30s} = {v:.4f}")


OOD pass: 60 examples

  EVAL PASS: ood  (n=60)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  50/60  |  10.90s/ex  |  ETA 1.8 min
  done in 10.8 min

Saved -> /kaggle/working/eval_reports/lora_v1_ood.json
  json_valid_pre_repair          = 1.0000
  json_valid_post_repair         = 1.0000
  schema_valid                   = 1.0000
  drug_precision                 = 1.0000
  drug_recall                    = 0.4697
  drug_f1                        = 0.6392
  ade_precision                  = 0.7742
  ade_recall                     = 0.6000
  ade_f1                         = 0.6761
  relation_f1                    = 0.5993
  hallucination_rate             = 0.0000
  evidence_accuracy              = 1.0000
  enum_accuracy                  = 1.0000
  span_f1_strict                 = 0.3871
  span_f1_lenient                = 0.8065
  latency_p50_s                  = 16.8861
  latency_p95_s                  = 19.9641


/tmp/ipykernel_58/1857834592.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow().isoformat() + "Z",


## Done

**Download** before session ends:
- `/kaggle/working/eval_reports/baseline.json`
- `/kaggle/working/eval_reports/lora_v1_ood.json`

Place both under `evaluation/reports/` in the repo. Then comparison.md can be generated locally.
